# Collaborator Data Access Guide — v1.1 Figures

This notebook documents **how collaborators should access the data used by each planned v1.1 KPI, ranking, and figure** in the Seattle Area Public Safety Dashboard.

The goal is not to reproduce every Plotly figure here. Instead, this notebook answers:

- Which production loader should I call?
- Which context key is authoritative for this metric/figure?
- Which identifier/date field should I use?
- Which population denominator should I use?
- Which data should **not** be substituted for the authoritative source?

This guide follows the current `staging` architecture and the contracts in:

- `docs/v1_1_metric_definitions.md`
- `docs/v1_1_figure_docket.md`
- `docs/dashboard_architecture.md`



## 1. Load the production dashboard contexts

The public access point for collaborators should be the same context loaders used by the dashboard.

- Crime: `load_crime_dashboard_context()`
- CAD/calls: `load_dashboard_context()`
- UOF/OIS: `load_uof_dashboard_context()`
- Population: already exposed through the crime and calls contexts, or directly through `load_dashboard_population()`

Avoid reading raw parquet files directly unless you are debugging the snapshot layer itself. The context loaders apply the production cleaning, classification, geography, and derived-data rules.

The cell below is a mandatory setup cell for notebooks not in the repo root (in a folder), copy paste it into new notebooks you make.

In [18]:
from pathlib import Path
import sys

# -------------------------------------------------------------------
# Locate repository root robustly whether Jupyter starts from repo root
# or from the notebooks directory.
# -------------------------------------------------------------------

cwd = Path.cwd().resolve()

repo_candidates = [cwd, *cwd.parents]

REPO_ROOT = next(
    (
        path
        for path in repo_candidates
        if (path / "dashboard").is_dir()
        and (path / "app.py").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root. "
        "Expected to find app.py and dashboard/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

Repository root: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard


In [19]:
from dashboard.crime_dashboard_data import load_crime_dashboard_context
from dashboard.spd_dashboard_data import load_dashboard_context
from dashboard.uof_dashboard_data import load_uof_dashboard_context
from dashboard.population_dashboard_data import load_dashboard_population

crime_context = load_crime_dashboard_context()
calls_context = load_dashboard_context()
uof_context = load_uof_dashboard_context()

neighborhood_population, city_population, population_metadata = load_dashboard_population()

### Context keys at a glance

**Crime context**

- `df` — full classified QA snapshot, including explicitly excluded rows.
- `valid_time` — authoritative analytical crime population with valid offense IDs/dates; includes offenses without map coordinates.
- `mappable_events` — analytical offenses with usable Seattle coordinates.
- `event_mcpp` — mappable offenses joined to spatial MCPP geography.
- `unmappable_events` — analytical offenses that cannot be represented as map points but may still contribute to analytical totals.
- `mcpp_boundaries` — MCPP polygons.
- `neighborhood_population` — calibrated MCPP population table.
- `city_population` — direct Seattle population estimate.
- `population_metadata` — ACS vintage/method metadata.

**Calls context**

- `df` — prepared dispatch-row snapshot.
- `valid_time` — dispatch rows with usable event IDs and queued timestamps.
- `event_mcpp` — coordinate-valid calls spatially matched to MCPPs.
- `response_analysis` — authoritative **event-level qualified response-time** table.
- `mcpp_boundaries` — MCPP polygons.
- `neighborhood_population`, `city_population`, `population_metadata` — population denominators and provenance.

**UOF context**

- `df` — full prepared UOF snapshot.
- `ois_events` — derived OIS events at the production day + normalized beat grain.
- `ois_rows` — source rows whose incident type matches OIS.
- `latest_available_date` — latest date in the UOF stream.
- `ois_rows_missing_event_date` — QA count for OIS rows that cannot form an event date.

In [20]:
print("Crime context keys:", sorted(crime_context))
print("Calls context keys:", sorted(calls_context))
print("UOF context keys:", sorted(uof_context))

Crime context keys: ['city_population', 'df', 'event_mcpp', 'event_mcpp_lookup', 'mappable_events', 'mcpp_boundaries', 'metadata', 'neighborhood_population', 'population_metadata', 'unmappable_events', 'valid_time', 'years_observed']
Calls context keys: ['city_population', 'df', 'event_mcpp', 'event_mcpp_lookup', 'mappable_events', 'mcpp_boundaries', 'metadata', 'neighborhood_population', 'population_metadata', 'response_analysis', 'valid_time', 'years_observed']
UOF context keys: ['df', 'latest_available_date', 'metadata', 'ois_events', 'ois_rows', 'ois_rows_missing_event_date']


## 2. Shared analysis-period helpers

For production-aligned analysis, use the shared period helpers rather than manually inventing date windows.

- `get_analysis_bounds(latest)` gives the selectable latest calendar-year domain.
- `get_previous_period(start, end, history_bounds=...)` gives the immediately adjacent equal-length comparison period.
- Crime filtering should use `filter_crime_records()` so category/subcategory/neighborhood/date behavior matches the dashboard.

In [21]:
import pandas as pd

from dashboard.analysis_windows import (
    get_analysis_bounds,
    get_history_bounds,
    get_previous_period,
)
from dashboard.crime_filters import filter_crime_records

## 3. Overall crime count KPI

**Authoritative source:** `crime_context["valid_time"]`

**Grain:** distinct `offense_id`

**Date field:** `offense_date`

Why `valid_time`? Citywide crime totals must include analytical offenses even when coordinates are missing or invalid. Do **not** count map points or use `event_mcpp` as the citywide analytical source.

In [22]:
crime = crime_context["valid_time"]

start_date = crime["offense_date"].max().normalize()
end_date = start_date
# FYI this is one day of data

state = {
    "start_date": str(start_date.date()),
    "end_date": str(end_date.date()),
    "crime_categories": [],
    "crime_subcategories": [],
    "neighborhoods": [],
}

selected_crime = filter_crime_records(crime, state)

overall_crime_count = selected_crime["offense_id"].nunique()
overall_crime_count

41

## 4. Overall crime rate KPI

**Numerator:** distinct `offense_id` from `crime_context["valid_time"]`

**Denominator:** `crime_context["city_population"]`

**Formula:** `count / city_population * 100_000`

Use the direct Seattle population value. Do **not** sum neighborhood populations and call that the city denominator.

In [23]:
crime_count = selected_crime["offense_id"].nunique()
city_population = crime_context["city_population"]

overall_crime_rate_per_100k = crime_count / city_population * 100_000
overall_crime_rate_per_100k

5.436259853220983

## 5. Top-level crime category counts

**Authoritative source:** `crime_context["valid_time"]`

**Category field:** `offense_category` (also mirrored into `event_importance_bin` for dashboard compatibility)

The current canonical analytical categories are:

- `crimes against persons`
- `crimes against property`
- `crimes against society / other`

In [24]:
category_counts = (
    selected_crime
    .groupby("offense_category")["offense_id"]
    .nunique()
    .sort_values(ascending=False)
)

category_counts

offense_category
crimes against property           17
crimes against society / other    13
crimes against persons            11
Name: offense_id, dtype: int64

## 6. Median qualified response-time KPI

**Authoritative source:** `calls_context["response_analysis"]`

**Grain:** one row per `cad_event_number`

**Date field:** `queued_time`

**Statistic:** median `response_time_minutes`

Do **not** derive this KPI from the calls map hover data or from raw dispatch rows. `response_analysis` is the production event-level qualified-response table.

In [25]:
response = calls_context["response_analysis"]

response_start = response["queued_time"].max().normalize()
response_end = response_start

response_selected = response.loc[
    pd.to_datetime(response["queued_time"]).dt.normalize().between(
        response_start, response_end
    )
].copy()

median_response_minutes = response_selected["response_time_minutes"].median()
median_response_minutes

np.float64(11.358333333333334)

## 7. UOF incident KPI

**Authoritative source:** `uof_context["df"]`

**Identifier:** distinct `incident_num`

**Date field:** `occured_date_time` (source spelling retained)

Use the production helper `count_uof_incidents()` rather than counting rows.

In [26]:
from dashboard.uof_dashboard_data import count_uof_incidents

uof_latest = uof_context["latest_available_date"]

if pd.notna(uof_latest):
    uof_day = str(uof_latest.date())
    uof_incident_count = count_uof_incidents(
        uof_context["df"],
        uof_day,
        uof_day,
    )
    print("UOF incidents:", uof_incident_count)

UOF incidents: 1


## 8. OIS event KPI

**Authoritative source:** `uof_context["ois_events"]`

**Identifier:** distinct `ois_event_key`

**Event definition:** Seattle-local calendar day + normalized beat

Use `count_ois_events()` rather than counting UOF rows or distinct force incident numbers.

In [27]:
from dashboard.uof_dashboard_data import count_ois_events

if pd.notna(uof_latest):
    ois_day = str(uof_latest.date())
    ois_event_count = count_ois_events(
        uof_context["ois_events"],
        ois_day,
        ois_day,
    )
    print("Derived OIS events:", ois_event_count)

Derived OIS events: 0


## 9. Neighborhood crime ranking

**Authoritative numerator source:** `crime_context["valid_time"]`

**Grouping field:** analytical `mcpp_neighborhood`

The analytical neighborhood field in `valid_time` uses spatial MCPP assignment when available and source neighborhood fallback otherwise. This is why rankings should be built from `valid_time`, not only from coordinate-valid map events.

For population-adjusted comparative rankings, join to `crime_context["neighborhood_population"]` and use the calibrated `population` field.

The current v1.1 methodology requires a `population >= 5_000` threshold for comparative rate ranking.

In [28]:
neighborhood_counts = (
    selected_crime
    .dropna(subset=["mcpp_neighborhood"])
    .groupby("mcpp_neighborhood")["offense_id"]
    .nunique()
    .rename("offense_count")
    .reset_index()
)

neighborhood_rates = neighborhood_counts.merge(
    crime_context["neighborhood_population"][
        ["mcpp_neighborhood", "population"]
    ],
    on="mcpp_neighborhood",
    how="left",
)

neighborhood_rates["crime_rate_per_100k"] = (
    neighborhood_rates["offense_count"]
    / neighborhood_rates["population"]
    * 100_000
)

comparative_rate_ranking = (
    neighborhood_rates.loc[neighborhood_rates["population"] >= 5_000]
    .sort_values("crime_rate_per_100k", ascending=False)
)

comparative_rate_ranking.head()

,mcpp_neighborhood,offense_count,population,crime_rate_per_100k
11,judkins park/north beacon hill,3,5372.0,55.845123
10,high point,4,9053.0,44.184248
5,chinatown/international district,2,5850.0,34.188034
6,claremont/rainier vista,2,6496.0,30.788177
13,miller park,2,8283.0,24.145841


## 10. Neighborhood response-time ranking

**Authoritative source:** `calls_context["response_analysis"]`

**Grouping field:** `dispatch_neighborhood`

**Statistic:** median `response_time_minutes`

This ranking should be calculated from qualified event-level records. Coordinates are not required.

> The minimum qualified-event sample size and tie policy are still methodology decisions for the ranking component. Do not silently reuse the current scatter plot's 100-event threshold as a ranking rule unless the methodology is explicitly updated.

In [29]:
response_ranking_source = response_selected.dropna(
    subset=["dispatch_neighborhood"]
).copy()

response_ranking = (
    response_ranking_source
    .groupby("dispatch_neighborhood")
    .agg(
        qualified_events=("cad_event_number", "nunique"),
        median_response_minutes=("response_time_minutes", "median"),
    )
    .reset_index()
    .sort_values("median_response_minutes")
)

response_ranking.tail()

,dispatch_neighborhood,qualified_events,median_response_minutes
31,magnolia,10,81.608333
58,wallingford,6,91.516667
50,sandpoint,12,92.950000
38,north admiral,4,99.141667
12,columbia city,4,154.791667


## 11. Crime choropleth (already exists/doesn't need significant revision)

For v1.1 analytical neighborhood shading, use:

- offense totals from `crime_context["valid_time"]`
- polygons from `crime_context["mcpp_boundaries"]`
- calibrated population from `crime_context["neighborhood_population"]`

The existing production map function currently has historical implementation differences, but the v1.1 analytical contract is to reconcile the choropleth with `valid_time` so coordinate-valid offenses without a spatial match can still use analytical neighborhood fallback.

For rates, use **per 100,000**, not per 1,000.

In [30]:
choropleth_data = (
    selected_crime
    .dropna(subset=["mcpp_neighborhood"])
    .groupby("mcpp_neighborhood")["offense_id"]
    .nunique()
    .rename("offense_count")
    .reset_index()
    .merge(
        crime_context["neighborhood_population"][
            ["mcpp_neighborhood", "population"]
        ],
        on="mcpp_neighborhood",
        how="left",
    )
)

choropleth_data["crime_rate_per_100k"] = (
    choropleth_data["offense_count"]
    / choropleth_data["population"]
    * 100_000
)

choropleth_data.tail()

,mcpp_neighborhood,offense_count,population,crime_rate_per_100k
17,pioneer square,1,2100.0,47.619048
18,rainier beach,1,5777.0,17.310023
19,sandpoint,1,38449.0,2.600848
20,slu/cascade,2,25464.0,7.854226
21,university,1,32535.0,3.073613


## 12. Crime point map (already exists/doesn't need significant revision)

**Authoritative point source:** `crime_context["event_mcpp"]`

Only coordinate-valid offenses can become points. The point layer is therefore intentionally narrower than the analytical population used for KPIs/rankings.

Use `crime_context["mcpp_boundaries"]` for polygon geometry.

Point-only filters such as free-text search should not change analytical neighborhood totals or choropleth shading.

In [31]:
crime_points = crime_context["event_mcpp"].copy()

point_columns = [
    "offense_id",
    "offense_date",
    "offense_category",
    "offense_sub_category",
    "mcpp_neighborhood",
    "latitude",
    "longitude",
]

crime_points[
    [column for column in point_columns if column in crime_points.columns]
].head()

,offense_id,offense_date,offense_category,offense_sub_category,mcpp_neighborhood,latitude,longitude
0,73116280560,2026-09-13 20:59:00,crimes against society / other,all other,sandpoint,47.664383,-122.283363
1,73114360114,2026-09-13 16:24:00,crimes against persons,assault offenses,fremont,47.662136,-122.350034
2,73113779421,2026-09-13 15:30:00,crimes against property,"property offenses (includes stolen, destruction)",capitol hill,47.620482,-122.322215
3,73114663039,2026-09-13 15:06:00,crimes against society / other,all other,capitol hill,47.617563,-122.322168
4,73113981059,2026-09-13 14:58:00,crimes against property,extortion/fraud/forgery/bribery (includes bad ...,pioneer square,47.599200,-122.330927


## 13. Crime daily time series (already exists/doesn't need significant revision)

**Authoritative source:** `crime_context["valid_time"]`

For dashboard-identical preparation, use `prepare_daily_event_data()` from `dashboard.crime_dashboard_figures` rather than manually resampling.

That helper handles the retained analysis domain and zero-filled daily series used by the figure.

In [32]:
from dashboard.crime_dashboard_figures import prepare_daily_event_data

selected_categories = sorted(
    crime_context["valid_time"]["offense_category"].dropna().unique()
)

daily_crime, crime_window = prepare_daily_event_data(
    crime_context,
    selected_categories,
    state,
)

daily_crime.head(), crime_window

(        date  reported_offenses  unique_reports  rolling_7_day_avg
 0 2025-09-13                188             167                NaN
 1 2025-09-14                180             157                NaN
 2 2025-09-15                199             179                NaN
 3 2025-09-16                191             173                NaN
 4 2025-09-17                181             163                NaN,
 {'earliest_available_day': Timestamp('2024-09-09 00:00:00'),
  'latest_available_day': Timestamp('2026-09-13 00:00:00'),
  'earliest_analysis_day': Timestamp('2025-09-13 00:00:00'),
  'plot_start_day': Timestamp('2025-09-13 00:00:00'),
  'plot_end_day': Timestamp('2026-09-13 00:00:00'),
  'initial_view_start': Timestamp('2026-09-12 00:00:00')})

## 14. Population access and provenance

Both the crime and calls contexts expose:

- `neighborhood_population`
- `city_population`
- `population_metadata`

The same values can be loaded directly with `load_dashboard_population()`.

Use:

- **direct city population** for Seattle-wide rates
- calibrated neighborhood `population` for MCPP rates
- `population_metadata` to display/record ACS vintage and methodology

Do not use `population_raw` as the runtime denominator.

In [36]:
display_columns = [
    "mcpp_neighborhood",
    "population",
    "population_raw",
    "population_year",
    "source_vintage",
    "estimation_method",
]

neighborhood_population[
    [c for c in display_columns if c in neighborhood_population.columns]
].head()

,mcpp_neighborhood,population,population_raw,population_year,source_vintage,estimation_method
0,alaska junction,16244.0,16243.0,2024,2024,ACS block-group population distributed using 2...
1,alki,7729.0,7728.0,2024,2024,ACS block-group population distributed using 2...
2,ballard north,30350.0,30348.0,2024,2024,ACS block-group population distributed using 2...
3,ballard south,26048.0,26046.0,2024,2024,ACS block-group population distributed using 2...
4,belltown,11501.0,11500.0,2024,2024,ACS block-group population distributed using 2...


## 15. Quick collaborator reference

| Component | Authoritative data access | Assignee |
| --- | --- | --- |
| Overall crime count | `crime_context["valid_time"]` | Ben Carr |
| Overall crime rate | `crime_context["valid_time"]` + `crime_context["city_population"]` | Parfait Ngandu |
| Crime category counts | `crime_context["valid_time"]` | Ben Carr |
| Median qualified response | `calls_context["response_analysis"]` | Ben Carr |
| UOF incidents | `uof_context["df"]` + `count_uof_incidents()` | Parfait Ngandu |
| OIS events | `uof_context["ois_events"]` + `count_ois_events()` | Parfait Ngandu |
| Neighborhood crime ranking | `crime_context["valid_time"]` + neighborhood population | Parfait Ngandu |
| Neighborhood response ranking | `calls_context["response_analysis"]` | Ben Carr |
| Crime choropleth | `crime_context["valid_time"]` + boundaries + population | Ben Carr |
| Crime point map | `crime_context["event_mcpp"]` | Ben Carr |
| Crime daily series | `crime_context["valid_time"]` via `prepare_daily_event_data()` | Ben Carr |




### Rule of thumb

- **Analysis/KPIs:** prefer `valid_time` or `response_analysis`.
- **Map points:** use `event_mcpp`.
- **QA/classification inspection:** use `df`.
- **Rates:** pair the analytical numerator with the appropriate production population denominator.
- **Do not substitute coordinate-valid records for the full analytical population.**

## 16. Optional sanity checks after loading

These checks are useful for collaborators to confirm that they are working with the intended production contexts before beginning analysis.

In [37]:
assert "valid_time" in crime_context
assert "response_analysis" in calls_context
assert "ois_events" in uof_context

assert crime_context["city_population"] > 0
assert calls_context["city_population"] > 0

assert crime_context["valid_time"]["offense_id"].notna().all()
assert calls_context["response_analysis"]["cad_event_number"].notna().all()

print("Core collaborator data-access checks passed.")

Core collaborator data-access checks passed.
